# 2. Premier modèle : TF-IDF + régression logistique

**Le problème** : un modèle ne comprend que des nombres. Il faut transformer chaque avis en une liste de nombres.

**L'idée de TF-IDF** : décrire un avis par **les mots qu'il contient**, en donnant plus de poids aux mots *parlants* (« génial », « nul ») qu'aux mots présents partout (« film »).

Ce notebook a 3 parties :
1. comprendre TF-IDF sur 3 petits avis (je l'ai écrit, tu lis et tu exécutes) ;
2. entraîner le modèle sur les 160 000 avis (**À TOI**) ;
3. regarder ce que le modèle a appris (**À TOI**).

## 2.1 Étape A : compter les mots (« sac de mots »)

On prend 3 mini-avis. `CountVectorizer` fait deux choses :
1. il **découpe** chaque avis en mots (la *tokenisation*), en mettant tout en minuscules et en enlevant la ponctuation ;
2. il **compte** combien de fois chaque mot apparaît dans chaque avis.

Résultat : un tableau avec **une ligne par avis** et **une colonne par mot du vocabulaire**.

In [21]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer

mini_avis = [
    "Film génial, acteurs géniaux !",
    "Film nul. Scénario nul.",
    "Pas mauvais, un film sympa.",
]

compteur = CountVectorizer()
X_compte = compteur.fit_transform(mini_avis)

pd.DataFrame(X_compte.toarray(), columns=compteur.get_feature_names_out())

,acteurs,film,génial,géniaux,mauvais,nul,pas,scénario,sympa,un
0,1,1,1,1,0,0,0,0,0,0
1,0,1,0,0,0,2,0,1,0,0
2,0,1,0,0,1,0,1,0,1,1


**Observe le tableau** :
- l'avis 2 a un **2** dans la colonne `nul` : le mot apparaît deux fois ;
- `génial` et `géniaux` sont deux colonnes **différentes** : pour l'ordinateur, ce sont deux mots sans rapport ;
- **l'ordre des mots a disparu** : « pas mauvais » devient juste « il y a *pas* » et « il y a *mauvais* ». C'est pour ça qu'on appelle cette méthode un **sac de mots** : on vide les mots dans un sac, on perd la phrase.

**Problème** : le mot `film` est dans les 3 avis. Il ne nous aide pas du tout à distinguer un avis positif d'un négatif, mais il compte autant que `génial`.

## 2.2 Étape B : pondérer les mots (TF-IDF)

TF-IDF corrige ce problème en multipliant deux choses :

- **TF** (*term frequency*) : combien de fois le mot est dans **cet** avis. C'est le comptage d'avant.
- **IDF** (*inverse document frequency*) : est-ce que le mot est **rare** dans l'ensemble des avis ? Un mot présent partout (`film`) reçoit un petit poids, un mot rare (`nul`) un grand poids.

**TF-IDF = TF × IDF**, puis chaque ligne est mise à la même échelle (pour qu'un avis long n'ait pas des nombres plus grands qu'un avis court).

In [22]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_mini = TfidfVectorizer()
X_tfidf = tfidf_mini.fit_transform(mini_avis)

pd.DataFrame(X_tfidf.toarray(), columns=tfidf_mini.get_feature_names_out()).round(2)

,acteurs,film,génial,géniaux,mauvais,nul,pas,scénario,sympa,un
0,0.55,0.32,0.55,0.55,0.00,0.00,0.00,0.00,0.00,0.00
1,0.00,0.26,0.00,0.00,0.00,0.86,0.00,0.43,0.00,0.00
2,0.00,0.28,0.00,0.00,0.48,0.00,0.48,0.00,0.48,0.48


In [23]:
# Le poids IDF de chaque mot : plus il est grand, plus le mot est rare
pd.Series(tfidf_mini.idf_, index=tfidf_mini.get_feature_names_out()).round(2)

acteurs     1.69
film        1.00
génial      1.69
géniaux     1.69
mauvais     1.69
nul         1.69
pas         1.69
scénario    1.69
sympa       1.69
un          1.69
dtype: float64

**Observe** :
- `film` a l'IDF le plus petit (1.0) car il est dans tous les avis ; dans le tableau, sa valeur est faible (≈ 0.3) ;
- `nul` vaut **0.86** dans l'avis 2 : il est rare **et** répété deux fois, c'est le mot le plus important de l'avis.

**Questions pour toi** (réponds dans la cellule de texte suivante) :
1. Pourquoi `film` a-t-il un poids plus faible que `génial` dans l'avis 1 ?
2. Dans l'avis 3, les mots `pas` et `mauvais` ont le même poids que `sympa`. Pourquoi est-ce un problème pour classer cet avis ?

**Mes réponses** :

1. car film ce repete dans les autres avis alors que genial non 
2. car on sait pas l'ordre "mauvais, pas sympa" ce n'est la meme chose que "pas mauvais, sympa" l'ordre est vraiment important.

## 2.3 À TOI : transformer les 160 000 avis

Maintenant on fait la même chose sur les vrais avis.

**Règle très importante : `fit` uniquement sur le train.**
- `fit_transform(train)` : le vectoriseur **apprend le vocabulaire et les IDF** sur le train, puis transforme le train ;
- `transform(val)` : il transforme la validation **avec le vocabulaire déjà appris**, sans rien réapprendre.

Si on faisait `fit` sur la validation, le modèle « verrait » des informations sur les données qu'on utilise pour le noter. C'est une **fuite de données** (*data leakage*), et la note serait faussement bonne.

**Exercice** :
1. Recharge `train` et `val` (comme au notebook 1, avec `pd.read_parquet`).
2. Crée `vectoriseur = TfidfVectorizer(max_features=50_000)`. (`max_features` garde seulement les 50 000 mots les plus fréquents, pour limiter la mémoire.)
3. Crée `X_train` avec `fit_transform` sur `train["review"]`, et `X_val` avec `transform` sur `val["review"]`.
4. Affiche `X_train.shape` : combien de lignes ? Combien de colonnes ? Est-ce logique ?

In [24]:
# À TOI : recharger les données
train=pd.read_parquet("data/train.parquet")
val=pd.read_parquet("data/validation.parquet")


In [25]:
# À TOI : créer le vectoriseur et transformer train et val
vectoriseur=TfidfVectorizer(max_features=50000)
X_train=vectoriseur.fit_transform(train["review"])
X_val=vectoriseur.transform(val["review"])
print(train.shape)
print(X_train.shape)

(160000, 2)
(160000, 50000)


160000 lignes et 50000 colonnes ce qui est logique car le nombre des avis deja present c'est 160000 ce qui est le nombre de ligne et dans les colonnes il y a 50000 c'est le nombre de mots qui sont presents dans notres avis limites car on a fait max_features=50000.

## 2.4 À TOI : entraîner la régression logistique

C'est le même modèle qu'au Titanic. La seule différence : au lieu de colonnes comme `Age` ou `Sex`, chaque colonne est maintenant un **mot**.

**Exercice** :
1. Crée `modele = LogisticRegression(max_iter=1000)` (importe-le depuis `sklearn.linear_model`).
2. Entraîne-le sur `X_train` et `train["label"]`.
3. Prédis sur `X_val`, puis calcule l'accuracy avec `accuracy_score` (depuis `sklearn.metrics`).

`max_iter=1000` : le modèle s'améliore par étapes successives ; on l'autorise à faire jusqu'à 1000 étapes pour qu'il ait le temps de converger.

**Question** : ton score est-il bien au-dessus du plancher de 50 % qu'on a trouvé au notebook 1 ?

In [26]:
# À TOI : entraîner et évaluer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
modele=LogisticRegression(max_iter=1000)
modele.fit(X_train,train["label"])
y_predict=modele.predict(X_val)
accuracy_score(val["label"],y_predict)


0.9253

**Mon score sur la validation** : 0.9253

*(On s'arrête ici pour cette séance. Dans la partie suivante : quels mots le modèle trouve-t-il les plus positifs et négatifs, où se trompe-t-il, et que donne-t-il sur tes avis pièges ?)*